# Energy Analytics Platform — Silver Pipeline

Lakeflow Spark Declarative Pipelines notebook source.

**Pipeline target:** `dbw_energy_analytics_dev.silver`

This notebook reads the existing Bronze tables. It does not modify Bronze data.

## Cell 1 — Imports and source configuration

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window


REGION_BRONZE = (
    "dbw_energy_analytics_dev.bronze.bronze_region_data"
)
FUEL_BRONZE = (
    "dbw_energy_analytics_dev.bronze.bronze_fuel_type_data"
)


## Cell 2 — Shared validation and snapshot functions

In [0]:
def add_quarantine_reason(df, required_key_columns):
    """
    Add a semicolon-separated reason for each invalid record.
    Original Bronze columns are retained.
    """

    value_as_double = F.col("value").cast("double")

    reasons = [
        F.when(
            F.col(column).isNull()
            | (F.trim(F.col(column)) == ""),
            F.lit(f"Missing or blank {column}")
        )
        for column in required_key_columns
    ]

    reasons.extend([
        F.when(
            F.col("pulled_at").isNull(),
            F.lit("Missing pulled_at")
        ),
        F.when(
            F.col("source_file").isNull()
            | (F.trim(F.col("source_file")) == ""),
            F.lit("Missing source_file")
        ),
        F.when(
            F.col("value").isNull()
            | (F.trim(F.col("value")) == ""),
            F.lit("Missing value")
        ),
        F.when(
            F.col("value").isNotNull()
            & (F.trim(F.col("value")) != "")
            & value_as_double.isNull(),
            F.lit("Value cannot be cast to DOUBLE")
        ),
        F.when(
            F.col("_rescued_data").isNotNull(),
            F.lit("Unexpected fields found in rescued data")
        )
    ])

    return df.withColumn(
        "quarantine_reason",
        F.concat_ws("; ", *reasons)
    )


def prepare_valid_snapshot(df, business_keys):
    """
    Build a complete cumulative snapshot from all Bronze history.

    For each business key, retain the latest observation by
    pulled_at. source_file is the deterministic tie-breaker.
    """

    valid_df = (
        df
        .filter(F.col("quarantine_reason") == "")
        .withColumn("value", F.col("value").cast("double"))
    )

    latest_record_window = (
        Window
        .partitionBy(*business_keys)
        .orderBy(
            F.col("pulled_at").desc(),
            F.col("source_file").desc()
        )
    )

    return (
        valid_df
        .withColumn(
            "_row_number",
            F.row_number().over(latest_record_window)
        )
        .filter(F.col("_row_number") == 1)
        .drop(
            "_row_number",
            "_rescued_data",
            "quarantine_reason"
        )
    )


## Cell 3 — Region quarantine

In [0]:
@dp.materialized_view(name="region_quarantine")
def region_quarantine():
    source_df = spark.read.table(REGION_BRONZE)

    validated_df = add_quarantine_reason(
        source_df,
        ["period", "respondent", "type"]
    )

    return validated_df.filter(
        F.col("quarantine_reason") != ""
    )


## Cell 4 — Region complete cumulative snapshot

In [0]:
@dp.materialized_view(name="region_snapshot")
def region_snapshot():
    source_df = spark.read.table(REGION_BRONZE)

    validated_df = add_quarantine_reason(
        source_df,
        ["period", "respondent", "type"]
    )

    return prepare_valid_snapshot(
        validated_df,
        ["period", "respondent", "type"]
    )


## Cell 5 — Fuel quarantine

In [0]:
@dp.materialized_view(name="fuel_quarantine")
def fuel_quarantine():
    source_df = spark.read.table(FUEL_BRONZE)

    validated_df = add_quarantine_reason(
        source_df,
        ["period", "respondent", "fueltype"]
    )

    return validated_df.filter(
        F.col("quarantine_reason") != ""
    )


## Cell 6 — Fuel complete cumulative snapshot

In [0]:
@dp.materialized_view(name="fuel_snapshot")
def fuel_snapshot():
    source_df = spark.read.table(FUEL_BRONZE)

    validated_df = add_quarantine_reason(
        source_df,
        ["period", "respondent", "fueltype"]
    )

    return prepare_valid_snapshot(
        validated_df,
        ["period", "respondent", "fueltype"]
    )


## Cell 7 — Region SCD Type 1 (current state)

In [0]:
dp.create_streaming_table(
    name="region_current"
)

dp.create_auto_cdc_from_snapshot_flow(
    target="region_current",
    source="region_snapshot",
    keys=["period", "respondent", "type"],
    stored_as_scd_type=1
)


## Cell 8 — Region SCD Type 2 (change history)

In [0]:
dp.create_streaming_table(
    name="region_history"
)

dp.create_auto_cdc_from_snapshot_flow(
    target="region_history",
    source="region_snapshot",
    keys=["period", "respondent", "type"],
    stored_as_scd_type=2,
    track_history_column_list=[
        "respondent_name",
        "type_name",
        "value",
        "value_units"
    ]
)


## Cell 9 — Fuel SCD Type 1 (current state)

In [0]:
dp.create_streaming_table(
    name="fuel_current"
)

dp.create_auto_cdc_from_snapshot_flow(
    target="fuel_current",
    source="fuel_snapshot",
    keys=["period", "respondent", "fueltype"],
    stored_as_scd_type=1
)


## Cell 10 — Fuel SCD Type 2 (change history)

In [0]:
dp.create_streaming_table(
    name="fuel_history"
)

dp.create_auto_cdc_from_snapshot_flow(
    target="fuel_history",
    source="fuel_snapshot",
    keys=["period", "respondent", "fueltype"],
    stored_as_scd_type=2,
    track_history_column_list=[
        "respondent_name",
        "type_name",
        "value",
        "value_units"
    ]
)
